In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pickle # lưu model sau khi training

# Import tf-idf và cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Import success!")

Import success!


I. Xây dựng thuật toán


In [2]:
def load_data():
    load_dotenv()
    try:
        connection_string = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_DATABASE')}"
        engine = create_engine(connection_string)
        print("Kết nối CSDL thành công!")
    except Exception as e:
        print(f"LỖI kết nối: {e}")
        return None, None

    # Lấy dữ liệu phim với các thông tin: genre, actor, director, summary để làm Item Profile
    sql_items = """
        SELECT 
            m.id, m.title, m.summary AS description,
            COALESCE(g.genres, '') AS genres,
            COALESCE(a.actors, '') AS actors,
            COALESCE(d.directors, '') AS directors
        FROM Movies m
        LEFT JOIN (
            SELECT movie_id, STRING_AGG(G.name, ', ') AS genres 
            FROM Movie_Genres MG JOIN Genres G ON MG.genre_id = G.id GROUP BY movie_id
        ) g ON m.id = g.movie_id
        LEFT JOIN (
            SELECT movie_id, STRING_AGG(A.name, ', ') AS actors 
            FROM Movie_Actors MA JOIN Actors A ON MA.actor_id = A.id GROUP BY movie_id
        ) a ON m.id = a.movie_id
        LEFT JOIN (
            SELECT movie_id, STRING_AGG(D.name, ', ') AS directors 
            FROM Movie_Directors MD JOIN Directors D ON MD.director_id = D.id GROUP BY movie_id
        ) d ON m.id = d.movie_id
    """

    # Lấy dữ liệu tương tác để làm User Profile
    sql_interactions = """
        SELECT *
        FROM view_movie_recommendation_scoring
    """

    try:
        print("Đang tải dữ liệu từ Postgres...")
        df_items = pd.read_sql(sql_items, engine)
        df_interactions = pd.read_sql(sql_interactions, engine)
        
        # Đảm bảo cột thời gian đúng định dạng để tính heuristic_score sau này
        df_interactions['last_watched_at'] = pd.to_datetime(df_interactions['last_watched_at'])
        
        print(f"Tải thành công: {len(df_items)} phim và {len(df_interactions)} tương tác.")
        return df_items, df_interactions

    except Exception as e:
        print(f"LỖI tải dữ liệu: {e}")
        return None, None

In [3]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        if isinstance(x, str):
            # Xóa khoảng trắng giữa các tên riêng (ví dụ: 'Action, Sci-Fi' -> 'action,sci-fi')
            # Sau đó xóa dấu phẩy để tạo thành chuỗi các từ đơn
            return x.replace(" ", "").lower().replace(",", " ")
        else:
            return ''

In [4]:
def create_soup(data_row):
    # Làm sạch các trường dữ liệu
    genres = clean_data(data_row['genres'])
    actors = clean_data(data_row['actors'])
    directors = clean_data(data_row['directors'])
    description = data_row['description'].lower() if data_row['description'] else ""

    
    return f"{genres} {actors} {directors} {description}"

In [5]:
df_items, df_interactions = load_data()
df_items.head()

Kết nối CSDL thành công!
Đang tải dữ liệu từ Postgres...
Tải thành công: 11440 phim và 12 tương tác.


,id,title,description,genres,actors,directors
0,5,Four Rooms,It's Ted the Bellhop's first night on the job....,Comedy,"Tim Roth, Jennifer Beals, David Proval, Ione S...",Allison Anders
1,11,Star Wars,Princess Leia is captured and held hostage by ...,"Adventure, Action, Science Fiction","Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas
2,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","Adventure, Animation, Family","Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton
3,13,Forrest Gump,A man with a low IQ has accomplished great thi...,"Drama, Comedy, Romance","Tom Hanks, Robin Wright, Gary Sinise, Sally Fi...",Robert Zemeckis
4,14,American Beauty,"Lester Burnham, a depressed suburban father in...",Drama,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes


In [17]:
def calculate_base_score(row, weights):
    v_progress = row['max_progress_percent'] / 100
    v_rating = row['rating'] / 5.0 if pd.notnull(row['rating']) else None
    v_fav = float(row['favourite'])
    v_watchlist = float(row['in_watchlist'])
    v_view_count = np.log(max(row['view_count'], 1)) / np.log(10)

    # Tính điểm theo từng thành phần hợp lệ
    numerator = (
        weights['progress'] * v_progress
        + weights['favourite'] * v_fav
        + weights['watchlist'] * v_watchlist
        + weights['view_count'] * v_view_count
    )
    denominator = (
        weights['progress']
        + weights['favourite']
        + weights['watchlist']
        + weights['view_count']
    )

    # Chỉ cộng rating khi có dữ liệu
    if v_rating is not None:
        numerator += weights['rating'] * v_rating
        denominator += weights['rating']

    return numerator / denominator * 10

In [18]:
weights = {
    'progress': 3.0,
    'rating': 4.1,
    'favourite': 4.5,
    'watchlist': 2.0,
    'view_count': 1.1
}

df_interactions['base_score'] = df_interactions.apply(calculate_base_score, axis=1, weights=weights)

In [19]:
df_interactions.head()


,user_id,movie_id,max_progress_percent,rating,view_count,favourite,in_watchlist,last_watched_at,base_score
0,3,278,0.859060,1.0,1,1,1,2026-03-09 22:38:28.348478,4.997124
1,3,1084242,0.006711,3.0,1,1,1,2026-03-09 22:38:28.348478,6.095375
2,7,1003596,0.917785,5.0,1,1,1,2026-03-09 22:38:28.348478,7.229615
3,7,299536,0.956376,4.0,1,1,1,2026-03-09 22:38:28.348478,6.672579
4,7,315635,0.870805,4.0,1,1,1,2026-03-09 22:38:28.348478,6.670833


In [20]:
# Tạo cột 'soup' bằng cách kết hợp các trường metadata 
df_items['soup'] = df_items.apply(create_soup, axis=1)
df_items.head()

,id,title,description,genres,actors,directors,soup
0,5,Four Rooms,It's Ted the Bellhop's first night on the job....,Comedy,"Tim Roth, Jennifer Beals, David Proval, Ione S...",Allison Anders,comedy timroth jenniferbeals davidproval iones...
1,11,Star Wars,Princess Leia is captured and held hostage by ...,"Adventure, Action, Science Fiction","Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,adventure action sciencefiction markhamill har...
2,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","Adventure, Animation, Family","Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton,adventure animation family albertbrooks ellend...
3,13,Forrest Gump,A man with a low IQ has accomplished great thi...,"Drama, Comedy, Romance","Tom Hanks, Robin Wright, Gary Sinise, Sally Fi...",Robert Zemeckis,drama comedy romance tomhanks robinwright gary...
4,14,American Beauty,"Lester Burnham, a depressed suburban father in...",Drama,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,drama kevinspacey annettebening thorabirch wes...


In [21]:
# Tính TF-IDF matrix cho cột 'soup'
vectorize = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorize.fit_transform(df_items['soup'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (11440, 61614)


In [22]:
#import SVD để giảm chiều dữ liệu
from sklearn.decomposition import TruncatedSVD

In [23]:
svd = TruncatedSVD(n_components=100, random_state=42)

latent_matrix = svd.fit_transform(tfidf_matrix)

print("Latent matrix shape after SVD:", latent_matrix.shape)

Latent matrix shape after SVD: (11440, 100)


In [24]:
# Lưu bảng tra cứu id -> index
indices = pd.Series(df_items.index, index=df_items['id']).drop_duplicates()
indices[299536]

6561

In [25]:
half_life_days = 30

lambda_val = np.log(2) / half_life_days

print("Lambda value for time decay:", lambda_val)

Lambda value for time decay: 0.023104906018664842


In [26]:
def build_user_profile(user_id, df_interactions, latent_matrix, indices, lambda_param):
    # Lọc tương tác của user
    user_interactions = df_interactions[df_interactions['user_id'] == user_id].copy()

    
    if user_interactions.empty:
        return None  # Trả về None nếu user chưa có tương tác nào
    
    # Lấy thời điểm mới nhất mà user xem để làm mốc dữ liệu
    ref_time = user_interactions['last_watched_at'].max()
    days_diff = (ref_time - user_interactions['last_watched_at']).dt.days


    # Tính điểm heuristic cho mỗi tương tác
    user_interactions['heuristic_score'] = user_interactions['base_score'] * np.exp(-lambda_param * days_diff)

    valid_movie_ids = set(indices.keys())

    # Lấy latent vectors của các phim đã xem 
    valid_interactions = user_interactions[user_interactions['movie_id'].isin(valid_movie_ids)]

    if valid_interactions.empty:
        return None  # Trả về None nếu không có phim nào trong latent matrix
    
    # Tính user profile
    movie_idxs = [indices[movie_id] for movie_id in valid_interactions['movie_id']]
    weights = valid_interactions['heuristic_score'].values.reshape(-1, 1)

    movie_vectors = latent_matrix[movie_idxs]
    user_profile = np.sum(movie_vectors * weights, axis=0) / np.sum(weights)

    return user_profile, valid_interactions

In [27]:
def recommendations_for_user(user_id, df_interactions, latent_matrix, indices, lambda_param, df_items, top_n=10):
    user_profile, valid_interactions = build_user_profile(user_id, df_interactions, latent_matrix, indices, lambda_param)
    
    if user_profile is None:
        print(f"User {user_id} chưa có tương tác hợp lệ nào.")
        return None
    
    # Lấy ra danh sách các movie_id đã xem
    seen_movie_ids = valid_interactions['movie_id'].unique()
    
    # Chuyển các movie_id đã xem sang index trong indices
    seen_movie_indices = [indices[movie_id] for movie_id in seen_movie_ids if movie_id in indices]
    
    # Tính cosine similarity giữa user profile và tất cả phim
    similarities = cosine_similarity(user_profile.reshape(1, -1), latent_matrix).flatten()
    
    # Loại bỏ các phim đã xem khỏi kết quả gợi ý
    similarities[seen_movie_indices] = -1
    
    # Lấy top N phim có similarity cao nhất
    top_indices = similarities.argsort()[-top_n:][::-1]
    
    recommendations = df_items.iloc[top_indices][['id', 'title', 'genres']]
    
    print(f"--- Top {top_n} gợi ý cho User {user_id} ---")
    return recommendations

In [45]:
rcm = recommendations_for_user(7, df_interactions, latent_matrix, indices, lambda_val, df_items, top_n=100)
print(rcm)

--- Top 100 gợi ý cho User 7 ---
          id                          title  \
6347  271110     Captain America: Civil War   
707     1726                       Iron Man   
8682  634649        Spider-Man: No Way Home   
1873   10138                     Iron Man 2   
6822  338676                     Extinction   
...      ...                            ...   
5914  197854  The Guyver: Bio-Booster Armor   
8806  665888                         Lapsis   
1635    9531                   Superman III   
5105   83533           Avatar: Fire and Ash   
9154  791042                         Levels   

                                                 genres  
6347                 Adventure, Action, Science Fiction  
707                  Adventure, Action, Science Fiction  
8682                 Adventure, Action, Science Fiction  
1873                 Adventure, Action, Science Fiction  
6822  Adventure, Horror, Action, Thriller, Science F...  
...                                                 ..

In [46]:
for movie in rcm['id']:
    print(movie)

271110
1726
634649
10138
338676
19543
940721
969681
290859
100402
5549
559
299534
442249
225914
102382
505642
8914
1979
15691
68721
39468
99861
1228710
450465
9362
8247
286567
324857
14372
8810
140607
760741
204634
557
283995
200560
493838
560016
102899
1225572
589304
187916
262500
10640
38050
9457
14235
10796
5137
616820
10529
400136
9825
549294
7191
429617
1309012
207768
168098
135397
823464
1083862
934433
1003598
44912
1196470
604
6593
1094274
1724
216527
1125510
1265609
31393
377264
1110034
479455
933131
241843
970450
348350
1300968
11850
762441
805320
1373445
68726
1033462
14869
852696
6479
1035048
520763
12636
197854
665888
9531
83533
791042


II. Sử dụng Precision@K, Recall@K, và F1-score để kiểm tra.
- Nguyên tắc: 1 phim được coi là liên quan đến user nếu số các thể loại user thích trong phim đó > số thể loại user không thích (Comparision of selected algorithms in Movie Recommender System)
- Cách tính: 
+ Precision@K: 

In [47]:
from collections import Counter

In [48]:
# Algorithm 1: Phân loại thể loại yêu thích và không thích của người dùng

def get_user_genres_preferences(user_id, df_interaction, df_items):
  # Get movieId which user rated
  user_ratings = df_interactions[df_interaction['user_id'] == user_id]

  # Count genre
  genre_counts = Counter()
  for m_id in user_ratings['movie_id']:
    # Get string genres of movie ("Action, Adventure")
    genre_str = df_items.loc[df_items['id'] == m_id, 'genres'].values[0]
    if pd.isna(genre_str): continue
    for genre in genre_str.split(', '):
      genre_counts[genre] += 1

  # Get all genre
  all_genres = set()
  df_items['genres'].str.split(', ').apply(lambda x: all_genres.update(x) if isinstance(x, list) else None)

  # Sort asc
  sorted_genres = sorted(list(all_genres), key=lambda x: genre_counts.get(x, 0), reverse=True)

  # Find mid-point
  mid_point = len(sorted_genres) // 2
  fav_genres = sorted_genres[:mid_point]
  unfav_genres = sorted_genres[mid_point:]

  return fav_genres, unfav_genres


In [49]:
# Algorithm 2: Kiểm tra tính liên quan (Relevance) dựa trên Genre
def is_movie_relevant(movie_id, df_movies, fav_genres, unfav_genres):
    res = df_movies.loc[df_movies['id'] == movie_id, 'genres'].values
    if len(res) == 0 or pd.isna(res[0]): return False

    movie_genres = [g.strip() for g in res[0].split(', ')]
    # Score = (Số genre thích) - (Số genre ghét)
    score = sum(1 for g in movie_genres if g in fav_genres) - \
            sum(1 for g in movie_genres if g in unfav_genres)
    return score > 0

In [50]:
def calculate_metric_at_k(user_id, recommended_movies):
    fav, unfav = get_user_genres_preferences(user_id, df_interactions, df_items)

    # Precision@K
    true_positives = sum(1 for movie in recommended_movies['id'] if is_movie_relevant(movie, df_items, fav, unfav))
    precision_at_k = true_positives / len(recommended_movies)

    return precision_at_k

In [51]:
calculate_metric_at_k(7, rcm)

0.99